In [1]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField , StructType , IntegerType , StringType , DataType
from pyspark.sql.functions import col

In [34]:
import configparser

config = configparser.ConfigParser()
config.read("/home/jovyan/work/connections.my_example_connection")

['/home/jovyan/work/connections.my_example_connection']

In [3]:
spark = SparkSession\
        .builder\
        .appName('LoadToSnowflake')\
        .master('local[*]')\
        .config("spark.jars.packages", "net.snowflake:spark-snowflake_2.12:2.15.0-spark_3.4,net.snowflake:snowflake-jdbc:3.14.4") \
        .getOrCreate()

In [35]:
sfURL = config['connections.my_example_connection']['sfURL'].strip('"')
sfUser = config['connections.my_example_connection']['sfUser'].strip('"')
sfPassword = config['connections.my_example_connection']['sfPassword'].strip('"')
sfDatabase = config['connections.my_example_connection']['sfDatabase'].strip('"')
sfSchema = config['connections.my_example_connection']['sfSchema'].strip('"')
sfWarehouse = config['connections.my_example_connection']['sfWarehouse'].strip('"')

In [37]:
sfOptions = {
    "sfURL": sfURL,
    "sfUser": sfUser,
    "sfPassword": sfPassword,
    "sfDatabase": sfDatabase,
    "sfSchema": sfSchema,
    "sfWarehouse": sfWarehouse,
}

In [20]:
base_path = "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/"

In [39]:
def load_table(table_name, mode="overwrite"):
    print(f"\n📤 Loading {table_name}...")
    df = spark.read.parquet(f"{base_path}/{table_name}")
    count = df.count()
    print(f"   Records: {count}")
    
    df.write \
        .format("snowflake") \
        .options(**sfOptions) \
        .option("dbtable", table_name.upper()) \
        .mode(mode) \
        .save()
    
    print(f"✅ {table_name} loaded")

In [40]:
try:
    load_table("dim_customers", "overwrite")
    load_table("dim_orders", "overwrite")
    load_table("dim_products", "overwrite")     
    load_table("dim_order_items", "overwrite")
    load_table("fact_sales", "overwrite")
except Exception as e:
    print(f"\n❌ LOADING FAILED: {e}")


📤 Loading dim_customers...
   Records: 100
✅ dim_customers loaded

📤 Loading dim_orders...
   Records: 300
✅ dim_orders loaded

📤 Loading dim_products...
   Records: 10
✅ dim_products loaded

📤 Loading dim_order_items...
   Records: 600
✅ dim_order_items loaded

📤 Loading fact_sales...
   Records: 600
✅ fact_sales loaded


In [ ]:
%%writefile big_data_pipline